# CV面试题 12: 编程实战题

本节包含 10 道计算机视觉面试中常见的编程实战题，涵盖：
- 图像几何变换 (旋转、仿射)
- 传统图像处理 (HOG, K-Means, GMM)
- 特征匹配与图像拼接
- 多分辨率融合
- 光流估计
- 图像超分辨率
- OCR 预处理流水线

每道题包含: 题目描述、代码框架、完整参考答案、面试要点。

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

---

## 实战题 1: 图像旋转 (任意角度)

**题目**: 不使用 cv2.rotate 或 scipy，手动实现图像的任意角度旋转。

**要求**:
- 构造仿射变换矩阵
- 使用逆映射避免空洞
- 实现双线性插值
- 旋转后图像能完整包含原图内容 (自动计算输出尺寸)

**输入**: 灰度图像 (H, W)，旋转角度 (度)

**输出**: 旋转后的图像

**面试要点**:
- 为什么用逆映射而不是正向映射？(正向映射会导致输出像素缺失和重复)
- 双线性插值的时间复杂度？O(H*W)
- 如何计算旋转后的画布大小？

In [ ]:
def rotate_image(image, angle_degrees):
    """
    手动实现图像任意角度旋转
    
    参数:
        image: 灰度图像 (H, W)
        angle_degrees: 旋转角度 (正数=逆时针)
    
    返回:
        rotated: 旋转后的图像
    """
    # TODO: 实现
    pass

In [ ]:
# ====== 参考答案 ======

def bilinear_interpolate(image, x, y):
    """双线性插值"""
    h, w = image.shape
    x0 = int(np.floor(x))
    y0 = int(np.floor(y))
    x1 = x0 + 1
    y1 = y0 + 1
    
    # 边界检查
    if x0 < 0 or x1 >= w or y0 < 0 or y1 >= h:
        return 0.0
    
    # 插值权重
    dx = x - x0
    dy = y - y0
    
    val = ((1-dx)*(1-dy)*image[y0, x0] + dx*(1-dy)*image[y0, x1] +
           (1-dx)*dy*image[y1, x0] + dx*dy*image[y1, x1])
    return val


def rotate_image(image, angle_degrees):
    """
    手动实现图像任意角度旋转
    
    核心步骤:
    1. 计算旋转矩阵
    2. 计算新画布大小
    3. 逆映射: 对输出每个像素找原图坐标
    4. 双线性插值
    """
    h, w = image.shape
    angle = np.radians(angle_degrees)
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    
    # 计算旋转后的画布大小
    corners = np.array([
        [0, 0], [w, 0], [w, h], [0, h]
    ], dtype=float)
    
    rot_corners = np.zeros_like(corners)
    for i, (cx, cy) in enumerate(corners):
        rot_corners[i] = [cos_a*cx - sin_a*cy, sin_a*cx + cos_a*cy]
    
    min_x = rot_corners[:, 0].min()
    max_x = rot_corners[:, 0].max()
    min_y = rot_corners[:, 1].min()
    max_y = rot_corners[:, 1].max()
    
    new_w = int(np.ceil(max_x - min_x))
    new_h = int(np.ceil(max_y - min_y))
    
    # 旋转矩阵的逆
    M_inv = np.array([
        [cos_a, sin_a],
        [-sin_a, cos_a]
    ])
    
    offset = np.array([-min_x, -min_y])
    rotated = np.zeros((new_h, new_w), dtype=image.dtype)
    
    for oy in range(new_h):
        for ox in range(new_w):
            rx, ry = ox - offset[0], oy - offset[1]
            sx = cos_a * rx + sin_a * ry
            sy = -sin_a * rx + cos_a * ry
            
            if 0 <= sx < w-1 and 0 <= sy < h-1:
                rotated[oy, ox] = bilinear_interpolate(image, sx, sy)
    
    return rotated


# === 测试 ===
size = 80
img = np.zeros((size, size), dtype=np.float64)
img[20:60, 15:65] = 0.5
img[30:50, 25:55] = 1.0
for i in range(size):
    if i < size:
        img[i, i] = 0.8

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('原图')

for idx, angle in enumerate([30, 45, 90]):
    rotated = rotate_image(img, angle)
    axes[idx+1].imshow(rotated, cmap='gray')
    axes[idx+1].set_title(f'旋转 {angle} 度\n({rotated.shape[1]}x{rotated.shape[0]})')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("面试要点:")
print("1. 逆映射 vs 正映射: 正映射会遗漏输出像素，逆映射保证每个输出像素都有值")
print("2. 双线性插值 O(1) per pixel, 总复杂度 O(H_out * W_out)")
print("3. 画布大小: 旋转后四角点的包围盒")
print("4. 向量化优化: 可以用 np.mgrid 批量计算坐标，避免双重循环")

---

## 实战题 2: 高斯混合模型 (GMM) 图像分割

**题目**: 使用 EM 算法实现 GMM 对图像进行分割。

**要求**:
- 实现 EM 算法 (E步: 计算后验概率, M步: 更新参数)
- 支持指定分量数 K
- 用于灰度图像的像素级分割

**输入**: 灰度图像 (H, W)，分量数 K

**输出**: 分割标签图 (H, W)，每个像素属于哪个分量

**面试要点**:
- EM 算法的收敛性？(保证单调不减，但可能陷入局部最优)
- GMM vs K-Means？(GMM是软聚类，可以建模不同形状的分布)
- 时间复杂度？O(N*K*iter)，N=像素数

In [ ]:
def gmm_segmentation(image, K=3, max_iter=50, tol=1e-4):
    """
    使用 GMM (EM算法) 进行图像分割
    
    参数:
        image: 灰度图像 (H, W), 值在 [0, 255] 或 [0, 1]
        K: 高斯分量数
        max_iter: 最大迭代次数
        tol: 收敛阈值
    
    返回:
        labels: 分割标签 (H, W), 值 0~K-1
        params: (means, variances, weights) GMM参数
    """
    # TODO: 实现
    pass

In [ ]:
# ====== 参考答案 ======

def gaussian_pdf(x, mean, var):
    """一维高斯概率密度"""
    return (1.0 / np.sqrt(2 * np.pi * var)) * np.exp(-0.5 * (x - mean)**2 / var)


def gmm_segmentation(image, K=3, max_iter=50, tol=1e-4):
    """
    GMM 图像分割 (EM 算法)
    
    EM 算法:
    E步: 计算每个像素属于每个分量的后验概率 (responsibility)
    M步: 根据后验概率更新均值、方差、权重
    """
    h, w = image.shape
    data = image.flatten().astype(np.float64)
    N = len(data)
    
    # 初始化参数
    np.random.seed(42)
    means = np.linspace(data.min(), data.max(), K)
    variances = np.ones(K) * (data.std() ** 2)
    weights = np.ones(K) / K
    
    log_likelihood_prev = -np.inf
    
    for iteration in range(max_iter):
        # === E步: 计算后验概率 ===
        r = np.zeros((N, K))
        for k in range(K):
            r[:, k] = weights[k] * gaussian_pdf(data, means[k], variances[k])
        
        r_sum = r.sum(axis=1, keepdims=True)
        r_sum = np.maximum(r_sum, 1e-10)
        r = r / r_sum
        
        log_likelihood = np.sum(np.log(r_sum + 1e-10))
        
        if abs(log_likelihood - log_likelihood_prev) < tol:
            break
        log_likelihood_prev = log_likelihood
        
        # === M步: 更新参数 ===
        N_k = r.sum(axis=0)
        means = (r.T @ data) / N_k
        
        for k in range(K):
            diff = data - means[k]
            variances[k] = (r[:, k] * diff**2).sum() / N_k[k]
            variances[k] = max(variances[k], 1e-6)
        
        weights = N_k / N
    
    labels = r.argmax(axis=1).reshape(h, w)
    return labels, (means, variances, weights)


# === 测试 ===
np.random.seed(42)
h, w = 150, 200
test_img = np.zeros((h, w))
test_img[:50, :] = 0.3 + np.random.randn(50, w) * 0.05
test_img[50:100, :] = 0.6 + np.random.randn(50, w) * 0.05
test_img[100:, :] = 0.9 + np.random.randn(50, w) * 0.05
test_img = np.clip(test_img, 0, 1)

labels, (means, vars_, weights) = gmm_segmentation(test_img, K=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img, cmap='gray')
axes[0].set_title('原始图像')
axes[1].imshow(labels, cmap='Set3')
axes[1].set_title('GMM分割结果 (K=3)')

sorted_idx = np.argsort(means)
reconstructed = np.zeros_like(test_img)
for new_label, old_label in enumerate(sorted_idx):
    reconstructed[labels == old_label] = means[old_label]
axes[2].imshow(reconstructed, cmap='gray')
axes[2].set_title('重建图像')

print(f"GMM参数:")
print(f"  均值: {means.round(4)}")
print(f"  方差: {vars_.round(6)}")
print(f"  权重: {weights.round(4)}")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("面试要点:")
print("1. EM算法: E步计算后验概率(软分配), M步更新参数; 交替优化直到收敛")
print("2. GMM vs K-Means: GMM是软聚类+概率模型, K-Means是硬聚类")
print("3. 时间复杂度: O(N*K*iter), N=像素数, K=分量数, iter=迭代次数")
print("4. 初始化敏感: 可用K-Means结果初始化GMM参数")

---

## 实战题 3: 非极大值抑制 (NMS) 完整实现

**题目**: 实现完整的多类别 NMS，支持批量和 Soft-NMS。

**要求**:
- 标准贪心 NMS
- Soft-NMS (高斯衰减)
- 多类别 NMS (每类独立执行)

**输入**: boxes (N,4), scores (N,), class_ids (N,)

**输出**: 保留的框索引

**面试要点**:
- NMS 的时间复杂度？O(N^2) 最坏情况
- Soft-NMS 的优势？(密集物体场景下不会误删相邻框)
- 为什么多类别 NMS 要独立执行？

In [ ]:
def compute_iou_vectorized(box, boxes):
    """计算一个框与一组框的 IoU"""
    # TODO
    pass

def nms_standard(boxes, scores, iou_threshold=0.5):
    """标准 NMS"""
    # TODO
    pass

def nms_soft(boxes, scores, sigma=0.5, score_threshold=0.01):
    """Soft-NMS (高斯衰减)"""
    # TODO
    pass

def multiclass_nms(boxes, scores, class_ids, num_classes, iou_threshold=0.5):
    """多类别 NMS"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def compute_iou_vectorized(box, boxes):
    """计算一个框与一组框的 IoU"""
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_box = (box[2] - box[0]) * (box[3] - box[1])
    area_boxes = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    union = area_box + area_boxes - inter
    
    return np.where(union > 0, inter / union, 0.0)


def nms_standard(boxes, scores, iou_threshold=0.5):
    """标准贪心 NMS"""
    boxes = np.array(boxes, dtype=np.float64)
    scores = np.array(scores, dtype=np.float64)
    
    order = scores.argsort()[::-1]
    keep = []
    
    while len(order) > 0:
        i = order[0]
        keep.append(int(i))
        
        if len(order) == 1:
            break
        
        ious = compute_iou_vectorized(boxes[i], boxes[order[1:]])
        remaining = np.where(ious <= iou_threshold)[0]
        order = order[remaining + 1]
    
    return keep


def nms_soft(boxes, scores, sigma=0.5, score_threshold=0.01):
    """Soft-NMS (高斯衰减)"""
    boxes = np.array(boxes, dtype=np.float64)
    scores = np.array(scores, dtype=np.float64).copy()
    
    N = len(boxes)
    indices = list(range(N))
    keep = []
    
    while len(indices) > 0:
        max_idx = max(indices, key=lambda i: scores[i])
        keep.append(max_idx)
        indices.remove(max_idx)
        
        if len(indices) == 0:
            break
        
        for j in indices:
            iou = compute_iou_vectorized(boxes[max_idx], boxes[j:j+1])[0]
            scores[j] *= np.exp(-(iou**2) / sigma)
        
        indices = [j for j in indices if scores[j] > score_threshold]
    
    return keep


def multiclass_nms(boxes, scores, class_ids, num_classes, iou_threshold=0.5):
    """多类别 NMS: 每个类别独立执行 NMS"""
    all_keep = []
    
    for cls_id in range(num_classes):
        mask = class_ids == cls_id
        if mask.sum() == 0:
            continue
        
        cls_boxes = boxes[mask]
        cls_scores = scores[mask]
        cls_indices = np.where(mask)[0]
        
        keep = nms_standard(cls_boxes, cls_scores, iou_threshold)
        all_keep.extend(cls_indices[keep].tolist())
    
    return all_keep


# === 测试 ===
np.random.seed(42)
N = 20
boxes = np.random.randint(0, 400, (N, 4)).astype(float)
boxes[:, 2] = boxes[:, 0] + np.random.randint(20, 100, N)
boxes[:, 3] = boxes[:, 1] + np.random.randint(20, 100, N)
scores = np.random.uniform(0.3, 0.99, N)
class_ids = np.random.randint(0, 3, N)

keep_std = nms_standard(boxes, scores, iou_threshold=0.5)
print(f"标准NMS: 保留 {len(keep_std)}/{N} 个框")

keep_soft = nms_soft(boxes, scores, sigma=0.5)
print(f"Soft-NMS: 保留 {len(keep_soft)}/{N} 个框")

keep_multi = multiclass_nms(boxes, scores, class_ids, num_classes=3, iou_threshold=0.5)
print(f"多类别NMS: 保留 {len(keep_multi)}/{N} 个框")

print("面试要点:")
print("1. 标准NMS: O(N^2) 最坏, 按分数贪心; 缺点: 密集物体可能误删")
print("2. Soft-NMS: 不删除而是衰减分数, 更适合密集场景")
print("3. 多类别NMS: 不同类别独立执行, 因为不同类的框重叠是合理的")
print("4. 其他变体: Weighted NMS, Softer NMS")

---

## 实战题 4: 图像拼接 (特征匹配 + 单应性矩阵)

**题目**: 实现简化版图像拼接。

**要求**:
1. 检测特征点 (Harris角点)
2. 特征描述与匹配
3. 使用 RANSAC 估计单应性矩阵
4. 图像变换与融合

**面试要点**:
- 单应性矩阵 H 有几个自由度？(8, 4对点求解)
- RANSAC 的原理和参数选择？
- 图像融合如何避免接缝？

In [ ]:
def harris_corner_detection(image, k=0.04, threshold=0.01):
    """Harris 角点检测"""
    # TODO
    pass

def compute_homography(src_pts, dst_pts):
    """用 DLT 计算单应性矩阵"""
    # TODO
    pass

def ransac_homography(src_pts, dst_pts, n_iter=1000, threshold=3.0):
    """RANSAC 估计单应性矩阵"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def harris_corner_detection(image, k=0.04, threshold=0.01):
    """Harris 角点检测"""
    Ix = ndimage.sobel(image, axis=1)
    Iy = ndimage.sobel(image, axis=0)
    
    Ix2 = ndimage.gaussian_filter(Ix * Ix, sigma=1.5)
    Iy2 = ndimage.gaussian_filter(Iy * Iy, sigma=1.5)
    IxIy = ndimage.gaussian_filter(Ix * Iy, sigma=1.5)
    
    det = Ix2 * Iy2 - IxIy**2
    trace = Ix2 + Iy2
    R = det - k * trace**2
    
    local_max = (R == ndimage.maximum_filter(R, size=5))
    corners = (R > threshold * R.max()) & local_max
    
    ys, xs = np.where(corners)
    responses = R[corners]
    return xs, ys, responses


def compute_homography(src_pts, dst_pts):
    """DLT 计算单应性矩阵"""
    n = src_pts.shape[0]
    A = np.zeros((2 * n, 9))
    
    for i in range(n):
        x, y = src_pts[i]
        xp, yp = dst_pts[i]
        A[2*i] = [-x, -y, -1, 0, 0, 0, xp*x, xp*y, xp]
        A[2*i+1] = [0, 0, 0, -x, -y, -1, yp*x, yp*y, yp]
    
    _, S, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    H = H / H[2, 2]
    return H


def ransac_homography(src_pts, dst_pts, n_iter=1000, threshold=3.0):
    """RANSAC 鲁棒估计单应性矩阵"""
    n = src_pts.shape[0]
    best_inliers = []
    best_H = None
    
    for _ in range(n_iter):
        idx = np.random.choice(n, 4, replace=False)
        H = compute_homography(src_pts[idx], dst_pts[idx])
        
        src_h = np.hstack([src_pts, np.ones((n, 1))])
        projected = (H @ src_h.T).T
        projected = projected[:, :2] / projected[:, 2:3]
        
        errors = np.linalg.norm(projected - dst_pts, axis=1)
        inliers = np.where(errors < threshold)[0]
        
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_H = H
    
    if len(best_inliers) >= 4:
        best_H = compute_homography(src_pts[best_inliers], dst_pts[best_inliers])
    
    return best_H, best_inliers


# === 测试 ===
np.random.seed(42)

angle = np.radians(5)
tx, ty = 50, 20
H_true = np.array([
    [np.cos(angle), -np.sin(angle), tx],
    [np.sin(angle), np.cos(angle), ty],
    [0, 0, 1]
])

h, w = 200, 300
n_pts = 50
src_pts = np.random.rand(n_pts, 2) * [w-50, h-50] + [25, 25]
src_h = np.hstack([src_pts, np.ones((n_pts, 1))])
dst_h = (H_true @ src_h.T).T
dst_pts = dst_h[:, :2] / dst_h[:, 2:3]
dst_pts += np.random.randn(n_pts, 2) * 1.0

outlier_mask = np.random.rand(n_pts) < 0.2
dst_pts[outlier_mask] += np.random.randn(outlier_mask.sum(), 2) * 30

H_est, inliers = ransac_homography(src_pts, dst_pts)
print(f"真实 H:\n{H_true}")
print(f"估计 H:\n{H_est}")
print(f"内点数: {len(inliers)}/{n_pts}")
print(f"H误差: {np.linalg.norm(H_est - H_true):.4f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.scatter(src_pts[:, 0], src_pts[:, 1], c='blue', label='源点', s=20)
ax.scatter(dst_pts[:, 0], dst_pts[:, 1], c='red', label='目标点', s=20, alpha=0.3)
ax.scatter(dst_pts[inliers, 0], dst_pts[inliers, 1], c='green', label='内点', s=30)
ax.legend()
ax.set_title('RANSAC 特征匹配结果')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("面试要点:")
print("1. 单应性矩阵H有8个自由度, 至少4对点求解")
print("2. RANSAC迭代次数 = log(1-p)/log(1-w^s)")
print("3. 融合方法: 线性加权融合, 在重叠区域做渐变过渡")

---

## 实战题 5: Laplacian 金字塔融合

**题目**: 实现 Laplacian 金字塔进行多分辨率图像融合。

**要求**:
1. 构建高斯金字塔
2. 构建拉普拉斯金字塔
3. 使用掩码在每层进行融合
4. 从金字塔重建图像

**面试要点**:
- 为什么用拉普拉斯金字塔而不是直接融合？(多尺度融合更自然，避免接缝)
- 高斯金字塔每层尺寸减半
- 拉普拉斯金字塔 = 当前层高斯 - 上层高斯上采样

In [ ]:
def gaussian_pyramid(image, levels=4):
    """构建高斯金字塔"""
    # TODO
    pass

def laplacian_pyramid(image, levels=4):
    """构建拉普拉斯金字塔"""
    # TODO
    pass

def pyramid_blend(img_a, img_b, mask, levels=4):
    """使用拉普拉斯金字塔融合两张图像"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def downsample(image):
    """下采样: 高斯模糊 + 尺寸减半"""
    blurred = ndimage.gaussian_filter(image, sigma=1.0)
    return blurred[::2, ::2]


def upsample(image, target_shape):
    """上采样到目标尺寸"""
    from scipy.ndimage import zoom
    factors = (target_shape[0] / image.shape[0], target_shape[1] / image.shape[1])
    return zoom(image, factors, order=1)


def gaussian_pyramid(image, levels=4):
    """构建高斯金字塔"""
    gp = [image.copy()]
    for _ in range(levels - 1):
        gp.append(downsample(gp[-1]))
    return gp


def laplacian_pyramid(image, levels=4):
    """构建拉普拉斯金字塔"""
    gp = gaussian_pyramid(image, levels)
    lp = []
    for i in range(len(gp) - 1):
        up = upsample(gp[i+1], gp[i].shape[:2])
        lp.append(gp[i] - up)
    lp.append(gp[-1])
    return lp


def reconstruct_from_pyramid(pyramid):
    """从拉普拉斯金字塔重建图像"""
    result = pyramid[-1]
    for i in range(len(pyramid) - 2, -1, -1):
        result = upsample(result, pyramid[i].shape[:2]) + pyramid[i]
    return result


def pyramid_blend(img_a, img_b, mask, levels=4):
    """拉普拉斯金字塔融合"""
    lp_a = laplacian_pyramid(img_a, levels)
    lp_b = laplacian_pyramid(img_b, levels)
    gp_mask = gaussian_pyramid(mask.astype(float), levels)
    
    blended = []
    for la, lb, gm in zip(lp_a, lp_b, gp_mask):
        gm_resized = gm[:la.shape[0], :la.shape[1]]
        if gm_resized.shape != la.shape:
            gm_resized = np.ones_like(la) * 0.5
        blended.append(gm_resized * la + (1 - gm_resized) * lb)
    
    return reconstruct_from_pyramid(blended)


# === 测试 ===
np.random.seed(42)
h, w = 128, 128
x = np.linspace(0, 1, w)
y = np.linspace(0, 1, h)
X, Y = np.meshgrid(x, y)

img_a = np.sin(2*np.pi*X) * np.cos(2*np.pi*Y)
img_a = (img_a + 1) / 2
img_b = np.exp(-((X-0.5)**2 + (Y-0.5)**2) / 0.1)

mask = np.zeros((h, w))
mask[:, :w//2] = 1.0
transition_width = 20
for i in range(transition_width):
    alpha = i / transition_width
    col = w//2 - transition_width//2 + i
    if 0 <= col < w:
        mask[:, col] = 1 - alpha

result = pyramid_blend(img_a, img_b, mask, levels=4)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(img_a, cmap='gray'); axes[0, 0].set_title('图像 A')
axes[0, 1].imshow(img_b, cmap='gray'); axes[0, 1].set_title('图像 B')
axes[0, 2].imshow(mask, cmap='gray'); axes[0, 2].set_title('融合掩码')
axes[1, 0].imshow(img_a * mask + img_b * (1-mask), cmap='gray')
axes[1, 0].set_title('直接融合 (有接缝)')
axes[1, 1].imshow(result, cmap='gray')
axes[1, 1].set_title('金字塔融合 (无接缝)')
axes[1, 2].imshow(np.abs(result - (img_a * mask + img_b * (1-mask))), cmap='hot')
axes[1, 2].set_title('融合差异')

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("面试要点:")
print("1. 高斯金字塔: 逐层高斯模糊+下采样, 保留低频信息")
print("2. 拉普拉斯金字塔: 高斯金字塔相邻层之差, 保留高频细节")
print("3. 融合优势: 在每层独立融合, 实现多尺度无缝过渡")
print("4. 时间复杂度: O(N), 因为金字塔总像素数约 4N/3")

---

## 实战题 6: HOG 特征提取

**题目**: 手动实现方向梯度直方图 (Histogram of Oriented Gradients) 特征提取。

**要求**:
1. 计算梯度 (幅值和方向)
2. 将图像分成 cell，统计方向直方图
3. 将 cell 组成 block，做对比度归一化
4. 拼接所有 block 的特征

**面试要点**:
- HOG 对光照变化鲁棒的原因？(梯度+归一化)
- Dalal-Triggs 方法的参数？(cell=8x8, block=2x2, 9个方向bin)
- 特征维度计算？

In [ ]:
def compute_gradients(image):
    """计算图像梯度 (幅值和方向)"""
    # TODO
    pass

def hog_features(image, cell_size=8, block_size=2, n_bins=9):
    """HOG 特征提取"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def compute_gradients(image):
    """计算图像梯度"""
    gx = np.zeros_like(image)
    gx[:, 1:-1] = image[:, 2:] - image[:, :-2]
    gx[:, 0] = image[:, 1] - image[:, 0]
    gx[:, -1] = image[:, -1] - image[:, -2]
    
    gy = np.zeros_like(image)
    gy[1:-1, :] = image[2:, :] - image[:-2, :]
    gy[0, :] = image[1, :] - image[0, :]
    gy[-1, :] = image[-1, :] - image[-2, :]
    
    magnitude = np.sqrt(gx**2 + gy**2)
    orientation = np.degrees(np.arctan2(gy, gx)) % 180
    
    return magnitude, orientation


def hog_features(image, cell_size=8, block_size=2, n_bins=9):
    """HOG 特征提取 (Dalal-Triggs)"""
    h, w = image.shape
    mag, ori = compute_gradients(image)
    
    n_cells_y = h // cell_size
    n_cells_x = w // cell_size
    bin_width = 180.0 / n_bins
    
    cell_hist = np.zeros((n_cells_y, n_cells_x, n_bins))
    
    for cy in range(n_cells_y):
        for cx in range(n_cells_x):
            cell_mag = mag[cy*cell_size:(cy+1)*cell_size,
                          cx*cell_size:(cx+1)*cell_size]
            cell_ori = ori[cy*cell_size:(cy+1)*cell_size,
                          cx*cell_size:(cx+1)*cell_size]
            
            for py in range(cell_size):
                for px in range(cell_size):
                    angle = cell_ori[py, px]
                    bin_idx = angle / bin_width
                    bin0 = int(bin_idx) % n_bins
                    bin1 = (bin0 + 1) % n_bins
                    weight1 = bin_idx - int(bin_idx)
                    weight0 = 1 - weight1
                    
                    cell_hist[cy, cx, bin0] += weight0 * cell_mag[py, px]
                    cell_hist[cy, cx, bin1] += weight1 * cell_mag[py, px]
    
    n_blocks_y = n_cells_y - block_size + 1
    n_blocks_x = n_cells_x - block_size + 1
    
    features = []
    for by in range(n_blocks_y):
        for bx in range(n_blocks_x):
            block = cell_hist[by:by+block_size, bx:bx+block_size, :].flatten()
            norm = np.sqrt(np.sum(block**2) + 1e-6**2)
            block_norm = block / norm
            block_norm = np.minimum(block_norm, 0.2)
            norm2 = np.sqrt(np.sum(block_norm**2) + 1e-6**2)
            block_norm = block_norm / norm2
            features.append(block_norm)
    
    return np.concatenate(features)


# === 测试 ===
h, w = 64, 64
test_img = np.zeros((h, w), dtype=np.float64)
test_img[20:22, 10:50] = 1.0
test_img[10:50, 30:32] = 1.0
for i in range(15, 50):
    test_img[i, i] = 1.0

features = hog_features(test_img, cell_size=8, block_size=2, n_bins=9)
print(f"图像尺寸: {test_img.shape}")
print(f"HOG 特征维度: {features.shape}")
print(f"理论维度: {7*7*4*9} = {(64//8-2+1)*(64//8-2+1)*2*2*9}")

mag, ori = compute_gradients(test_img)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(test_img, cmap='gray'); axes[0].set_title('原图')
axes[1].imshow(mag, cmap='hot'); axes[1].set_title('梯度幅值')
axes[2].imshow(ori, cmap='hsv'); axes[2].set_title('梯度方向')
axes[3].bar(range(9), features[:9])
axes[3].set_title('第一个block直方图')
axes[3].set_xlabel('方向 bin')
axes[3].set_ylabel('归一化幅值')
for ax in axes[:3]:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("面试要点:")
print("1. HOG参数: cell=8x8, block=2x2cells, 9方向bins")
print("2. 特征维度: (W/8-1)*(H/8-1)*4*9 (64x128图像=3780维)")
print("3. 对光照鲁棒: 梯度消除加性变化, block归一化消除乘性变化")
print("4. 双线性插值: 角度在相邻bin按比例分配幅值")

---

## 实战题 7: K-Means 图像分割

**题目**: 基于 K-Means 聚类实现图像颜色分割。

**要求**:
- 支持灰度和彩色图像
- 实现 K-Means++ 初始化
- 可选择颜色特征或颜色+位置特征

**面试要点**:
- K-Means 的时间复杂度？O(N*K*iter*d)
- K-Means++ 初始化为什么更好？
- 如何选择 K？(肘部法则, 轮廓系数)

In [ ]:
def kmeans_pp_init(data, K):
    """K-Means++ 初始化"""
    # TODO
    pass

def kmeans_segmentation(image, K=3, max_iter=100, use_position=False):
    """K-Means 图像分割"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def kmeans_pp_init(data, K):
    """K-Means++ 初始化"""
    n = data.shape[0]
    centers = np.zeros((K, data.shape[1]))
    
    idx = np.random.randint(n)
    centers[0] = data[idx]
    
    for k in range(1, K):
        dists = np.zeros(n)
        for j in range(k):
            d = np.sum((data - centers[j])**2, axis=1)
            dists = np.maximum(dists, d) if j > 0 else d
        
        probs = dists / dists.sum()
        idx = np.random.choice(n, p=probs)
        centers[k] = data[idx]
    
    return centers


def kmeans_segmentation(image, K=3, max_iter=100, use_position=False):
    """K-Means 图像分割"""
    h, w = image.shape[:2]
    
    if image.ndim == 2:
        features = image.flatten().reshape(-1, 1)
    else:
        features = image.reshape(-1, image.shape[2])
    
    if use_position:
        yy, xx = np.mgrid[0:h, 0:w]
        pos = np.stack([xx.flatten() / w, yy.flatten() / h], axis=1)
        features = np.hstack([features, pos])
    
    np.random.seed(42)
    centers = kmeans_pp_init(features, K)
    
    for iteration in range(max_iter):
        dists = np.array([np.sum((features - centers[k])**2, axis=1) for k in range(K)])
        labels = dists.argmin(axis=0)
        
        new_centers = np.zeros_like(centers)
        for k in range(K):
            mask = labels == k
            if mask.sum() > 0:
                new_centers[k] = features[mask].mean(axis=0)
            else:
                new_centers[k] = centers[k]
        
        if np.allclose(centers, new_centers, atol=1e-6):
            break
        centers = new_centers
    
    return labels.reshape(h, w), centers


# === 测试 ===
np.random.seed(42)
h, w = 120, 160
test_img = np.zeros((h, w, 3))
test_img[:40, :, 0] = 0.8
test_img[:40, :, 1] = 0.2
test_img[40:80, :, 1] = 0.8
test_img[80:, :, 2] = 0.8
test_img += np.random.randn(h, w, 3) * 0.05
test_img = np.clip(test_img, 0, 1)

labels, centers = kmeans_segmentation(test_img, K=3)

reconstructed = np.zeros_like(test_img)
for k in range(3):
    mask = labels == k
    reconstructed[mask] = centers[k, :3]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img); axes[0].set_title('原始图像')
axes[1].imshow(labels, cmap='Set3'); axes[1].set_title('K-Means分割')
axes[2].imshow(np.clip(reconstructed, 0, 1)); axes[2].set_title('分割重建')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("面试要点:")
print("1. 时间复杂度: O(N*K*iter*d)")
print("2. K-Means++: 初始中心尽量分散")
print("3. 选K方法: 肘部法则, 轮廓系数, Gap Statistic")
print("4. 加入位置特征可以鼓励空间连续性")

---

## 实战题 8: 光流估计 (Lucas-Kanade)

**题目**: 实现 Lucas-Kanade 光流估计算法。

**原理**: 光流约束方程 Ix*u + Iy*v + It = 0，LK方法假设局部区域光流一致。

**要求**:
1. 计算时空梯度 (Ix, Iy, It)
2. 在每个像素的邻域构建方程
3. 使用最小二乘求解

**面试要点**:
- LK 方法的假设？(亮度恒定, 小运动, 空间一致性)
- 金字塔 LK 的作用？(处理大运动)
- Horn-Schunck 与 LK 的区别？(全局平滑 vs 局部一致)

In [ ]:
def lucas_kanade(img1, img2, window_size=15):
    """Lucas-Kanade 光流估计"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def lucas_kanade(img1, img2, window_size=15):
    """Lucas-Kanade 光流估计"""
    h, w = img1.shape
    
    Ix = ndimage.sobel(img1, axis=1) * 0.5
    Iy = ndimage.sobel(img1, axis=0) * 0.5
    It = img2.astype(np.float64) - img1.astype(np.float64)
    
    kernel = np.ones((window_size, window_size))
    
    Ix2 = ndimage.convolve(Ix * Ix, kernel)
    Iy2 = ndimage.convolve(Iy * Iy, kernel)
    IxIy = ndimage.convolve(Ix * Iy, kernel)
    IxIt = ndimage.convolve(Ix * It, kernel)
    IyIt = ndimage.convolve(Iy * It, kernel)
    
    det = Ix2 * Iy2 - IxIy**2
    valid = np.abs(det) > 1e-6
    
    flow_x = np.zeros((h, w))
    flow_y = np.zeros((h, w))
    
    flow_x[valid] = -(Iy2[valid] * IxIt[valid] - IxIy[valid] * IyIt[valid]) / det[valid]
    flow_y[valid] = -(Ix2[valid] * IyIt[valid] - IxIy[valid] * IxIt[valid]) / det[valid]
    
    return flow_x, flow_y


# === 测试 ===
np.random.seed(42)
h, w = 100, 120
x = np.linspace(0, 4*np.pi, w)
y = np.linspace(0, 4*np.pi, h)
X, Y = np.meshgrid(x, y)

img1 = (np.sin(X) * np.cos(Y) + 1) / 2
true_dx, true_dy = 2.0, 1.0
img2 = (np.sin(X - true_dx * (4*np.pi/w)) * np.cos(Y - true_dy * (4*np.pi/h)) + 1) / 2

flow_x, flow_y = lucas_kanade(img1, img2, window_size=15)

valid_mask = (np.abs(flow_x) > 0) | (np.abs(flow_y) > 0)
if valid_mask.sum() > 0:
    print(f"真实光流: dx={true_dx}, dy={true_dy}")
    print(f"估计光流: dx={flow_x[valid_mask].mean():.2f}, dy={flow_y[valid_mask].mean():.2f}")

# 可视化
step = 5
yy, xx = np.mgrid[0:h:step, 0:w:step]
fx = flow_x[::step, ::step]
fy = flow_y[::step, ::step]
mag = np.sqrt(fx**2 + fy**2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.sqrt(flow_x**2 + flow_y**2), cmap='hot')
axes[0].set_title('光流幅值')
axes[1].imshow(np.zeros((h, w)), cmap='gray', alpha=0.3)
axes[1].quiver(xx, yy, fx, fy, mag, cmap='hsv', scale=1, scale_units='xy', angles='xy')
axes[1].set_title('光流向量场')
axes[1].set_aspect('equal')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

print("面试要点:")
print("1. LK三大假设: 亮度恒定, 小运动, 空间一致性")
print("2. 金字塔LK: 先在低分辨率估计大位移, 再逐步细化")
print("3. Horn-Schunck: 全局平滑约束; LK: 局部约束")
print("4. 时间复杂度: O(H*W) (用积分图像优化)")

---

## 实战题 9: 图像超分辨率 (双三次插值)

**题目**: 实现双三次插值 (Bicubic Interpolation) 进行图像放大。

**要求**:
- 实现双三次插值核 (cubic convolution kernel)
- 支持任意整数倍放大
- 与双线性插值对比效果

**面试要点**:
- 双三次 vs 双线性？(双三次考虑4x4邻域，更锐利但更慢)
- 插值核函数的选择？(a=-0.5 或 a=-0.75)
- 时间复杂度？O(H_out * W_out * 16)

In [ ]:
def cubic_kernel(x, a=-0.5):
    """双三次插值核函数"""
    # TODO
    pass

def bicubic_interpolate(image, scale):
    """双三次插值放大图像"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def cubic_kernel(x, a=-0.5):
    """
    双三次插值核
    W(x) = (a+2)|x|^3 - (a+3)|x|^2 + 1,    0<=|x|<=1
         = a|x|^3 - 5a|x|^2 + 8a|x| - 4a,    1<|x|<=2
         = 0,                                   |x|>2
    """
    x = np.abs(x)
    result = np.zeros_like(x)
    
    mask1 = x <= 1
    result[mask1] = (a + 2) * x[mask1]**3 - (a + 3) * x[mask1]**2 + 1
    
    mask2 = (x > 1) & (x <= 2)
    result[mask2] = a * x[mask2]**3 - 5*a * x[mask2]**2 + 8*a * x[mask2] - 4*a
    
    return result


def bicubic_interpolate(image, scale):
    """双三次插值"""
    h, w = image.shape
    new_h, new_w = int(h * scale), int(w * scale)
    output = np.zeros((new_h, new_w))
    
    for oy in range(new_h):
        for ox in range(new_w):
            sx = (ox + 0.5) / scale - 0.5
            sy = (oy + 0.5) / scale - 0.5
            
            x0 = int(np.floor(sx))
            y0 = int(np.floor(sy))
            
            value = 0.0
            weight_sum = 0.0
            
            for dy in range(-1, 3):
                for dx in range(-1, 3):
                    nx = x0 + dx
                    ny = y0 + dy
                    
                    if 0 <= nx < w and 0 <= ny < h:
                        wx = cubic_kernel(sx - nx)
                        wy = cubic_kernel(sy - ny)
                        w_total = wx * wy
                        value += w_total * image[ny, nx]
                        weight_sum += w_total
            
            if weight_sum > 0:
                output[oy, ox] = value / weight_sum
    
    return output


def bilinear_interpolate_simple(image, scale):
    """双线性插值 (对比)"""
    h, w = image.shape
    new_h, new_w = int(h * scale), int(w * scale)
    output = np.zeros((new_h, new_w))
    
    for oy in range(new_h):
        for ox in range(new_w):
            sx = (ox + 0.5) / scale - 0.5
            sy = (oy + 0.5) / scale - 0.5
            
            x0 = max(int(np.floor(sx)), 0)
            y0 = max(int(np.floor(sy)), 0)
            x1 = min(x0 + 1, w - 1)
            y1 = min(y0 + 1, h - 1)
            
            dx = sx - x0
            dy = sy - y0
            
            output[oy, ox] = ((1-dx)*(1-dy)*image[y0,x0] + dx*(1-dy)*image[y0,x1] +
                              (1-dx)*dy*image[y1,x0] + dx*dy*image[y1,x1])
    return output


# === 测试 ===
np.random.seed(42)
h, w = 32, 48
x = np.linspace(0, 2*np.pi, w)
y = np.linspace(0, 2*np.pi, h)
X, Y = np.meshgrid(x, y)
test_img = (np.sin(3*X) * np.cos(2*Y) + 1) / 2

scale = 3
bicubic_result = bicubic_interpolate(test_img, scale)
bilinear_result = bilinear_interpolate_simple(test_img, scale)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img, cmap='gray')
axes[0].set_title(f'原图 ({h}x{w})')
axes[1].imshow(bilinear_result, cmap='gray')
axes[1].set_title(f'双线性 x{scale}')
axes[2].imshow(bicubic_result, cmap='gray')
axes[2].set_title(f'双三次 x{scale}')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

diff = np.abs(bicubic_result - bilinear_result)
plt.figure(figsize=(6, 5))
plt.imshow(diff, cmap='hot')
plt.colorbar(label='|双三次 - 双线性|')
plt.title('插值方法差异')
plt.axis('off')
plt.show()

print("面试要点:")
print("1. 双三次考虑4x4=16个邻域像素, 双线性只用2x2=4个")
print("2. 双三次更锐利但可能有过冲, 双线性更平滑但模糊")
print("3. 时间复杂度: 双三次 O(H_out*W_out*16), 双线性 O(H_out*W_out*4)")
print("4. 参数a: -0.5(Keys), -0.75(锐利), -1.0(Catmull-Rom)")

---

## 实战题 10: OCR 预处理流水线

**题目**: 实现完整的 OCR 预处理流水线。

**要求**:
1. 灰度化
2. 二值化 (大津法/自适应)
3. 去噪 (中值滤波/形态学操作)
4. 倾斜校正 (投影法)
5. 字符分割 (连通域分析)

**面试要点**:
- 每步的目的和可选方法？
- 大津法的原理？(最大化类间方差)
- 倾斜校正的替代方法？(霍夫变换, 投影法, 基于轮廓)

In [ ]:
def to_grayscale(image):
    """灰度化"""
    # TODO
    pass

def otsu_threshold(image):
    """大津法自动阈值"""
    # TODO
    pass

def deskew(binary_image):
    """倾斜校正"""
    # TODO
    pass

def connected_components(binary_image):
    """简单连通域分析"""
    # TODO
    pass

def ocr_preprocess(image):
    """完整OCR预处理流水线"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def to_grayscale(image):
    """灰度化: Y = 0.299R + 0.587G + 0.114B"""
    if image.ndim == 2:
        return image.copy()
    return 0.299 * image[:,:,0] + 0.587 * image[:,:,1] + 0.114 * image[:,:,2]


def otsu_threshold(image):
    """
    大津法: 遍历所有阈值, 找最大类间方差
    sigma_b^2(t) = w0*w1*(mu0-mu1)^2
    """
    hist = np.bincount(image.flatten().astype(int), minlength=256)
    total = image.size
    
    best_threshold = 0
    best_variance = 0
    
    sum_total = np.sum(np.arange(256) * hist)
    sum_bg = 0
    weight_bg = 0
    
    for t in range(256):
        weight_bg += hist[t]
        if weight_bg == 0:
            continue
        weight_fg = total - weight_bg
        if weight_fg == 0:
            break
        
        sum_bg += t * hist[t]
        mean_bg = sum_bg / weight_bg
        mean_fg = (sum_total - sum_bg) / weight_fg
        
        variance = weight_bg * weight_fg * (mean_bg - mean_fg)**2
        if variance > best_variance:
            best_variance = variance
            best_threshold = t
    
    binary = (image > best_threshold).astype(np.uint8)
    return binary, best_threshold


def median_denoise(image, kernel_size=3):
    """中值滤波去噪"""
    return ndimage.median_filter(image, size=kernel_size)


def deskew(binary_image):
    """倾斜校正 (投影法: 找使水平投影方差最大的角度)"""
    best_angle = 0
    best_score = 0
    
    for angle in np.arange(-10, 10, 0.5):
        rotated = ndimage.rotate(binary_image.astype(float), angle, reshape=False)
        projection = rotated.sum(axis=1)
        score = projection.var()
        if score > best_score:
            best_score = score
            best_angle = angle
    
    corrected = ndimage.rotate(binary_image.astype(float), best_angle, reshape=False)
    return corrected, best_angle


def connected_components(binary_image):
    """连通域标记 (BFS)"""
    h, w = binary_image.shape
    labels = np.zeros((h, w), dtype=int)
    current_label = 0
    
    for y in range(h):
        for x in range(w):
            if binary_image[y, x] == 0 or labels[y, x] > 0:
                continue
            
            current_label += 1
            queue = [(y, x)]
            labels[y, x] = current_label
            
            while queue:
                cy, cx = queue.pop(0)
                for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
                    ny, nx = cy+dy, cx+dx
                    if (0 <= ny < h and 0 <= nx < w and
                        binary_image[ny, nx] > 0 and labels[ny, nx] == 0):
                        labels[ny, nx] = current_label
                        queue.append((ny, nx))
    
    return labels, current_label


def segment_characters(binary_image, labels, n_components):
    """从连通域中分割字符"""
    chars = []
    h, w = binary_image.shape
    
    for label_id in range(1, n_components + 1):
        ys, xs = np.where(labels == label_id)
        if len(xs) < 10 or len(xs) > h * w * 0.5:
            continue
        
        margin = 2
        x1 = max(0, xs.min() - margin)
        y1 = max(0, ys.min() - margin)
        x2 = min(w-1, xs.max() + margin)
        y2 = min(h-1, ys.max() + margin)
        chars.append((x1, y1, x2, y2))
    
    chars.sort(key=lambda b: b[0])
    return chars


def ocr_preprocess(image):
    """完整 OCR 预处理流水线"""
    results = {}
    
    gray = to_grayscale(image)
    gray = (gray / gray.max() * 255).astype(np.uint8)
    results['gray'] = gray
    
    binary, threshold = otsu_threshold(gray)
    results['binary'] = binary
    results['threshold'] = threshold
    
    denoised = median_denoise(binary, kernel_size=3)
    results['denoised'] = denoised
    
    corrected, angle = deskew(denoised)
    results['corrected'] = (corrected > 0.5).astype(np.uint8)
    results['skew_angle'] = angle
    
    labels, n_comp = connected_components(results['corrected'])
    chars = segment_characters(results['corrected'], labels, n_comp)
    results['labels'] = labels
    results['characters'] = chars
    
    return results


# === 测试 ===
np.random.seed(42)
h, w = 100, 300
text_img = np.ones((h, w), dtype=np.uint8) * 255

chars_positions = [20, 50, 80, 110, 150, 180, 210, 240]
for cx in chars_positions:
    cw = np.random.randint(12, 25)
    ch = np.random.randint(20, 35)
    cy = 30 + np.random.randint(-5, 10)
    text_img[cy:cy+ch, cx:cx+cw] = 0

noise = np.random.rand(h, w) > 0.97
text_img[noise] = 0
text_img = ndimage.rotate(text_img, 2, reshape=False, order=0)

results = ocr_preprocess(text_img)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0, 0].imshow(text_img, cmap='gray')
axes[0, 0].set_title('原始图像 (含噪声+倾斜)')
axes[0, 1].imshow(results['gray'], cmap='gray')
axes[0, 1].set_title('灰度化')
axes[0, 2].imshow(results['binary'], cmap='gray')
axes[0, 2].set_title(f"二值化 (阈值={results['threshold']})")
axes[1, 0].imshow(results['denoised'], cmap='gray')
axes[1, 0].set_title('去噪 (中值滤波)')
axes[1, 1].imshow(results['corrected'], cmap='gray')
axes[1, 1].set_title(f"倾斜校正 (角度={results['skew_angle']:.1f}度)")

seg_img = np.stack([results['corrected']*255]*3, axis=-1).astype(np.uint8)
for (x1, y1, x2, y2) in results['characters']:
    seg_img[y1:y2, x1] = [255, 0, 0]
    seg_img[y1:y2, x2] = [255, 0, 0]
    seg_img[y1, x1:x2] = [255, 0, 0]
    seg_img[y2, x1:x2] = [255, 0, 0]
axes[1, 2].imshow(seg_img)
axes[1, 2].set_title(f"字符分割 ({len(results['characters'])}个字符)")

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"检测到 {len(results['characters'])} 个字符区域")
print(f"倾斜角度: {results['skew_angle']:.1f} 度")
print(f"Otsu阈值: {results['threshold']}")

print("面试要点:")
print("1. 灰度化: Y = 0.299R + 0.587G + 0.114B, 绿色权重最大")
print("2. 大津法: 遍历0-255找最大类间方差, O(256*L)")
print("3. 去噪: 中值滤波(椒盐噪声), 高斯滤波(高斯噪声)")
print("4. 倾斜校正: 霍夫变换(检测直线角度), 投影法(最大化水平投影方差)")
print("5. 字符分割: 连通域分析 -> 过滤噪声 -> 外接矩形 -> 按x排序")

---

## 综合总结

| 编号 | 题目 | 核心算法 | 时间复杂度 |
|------|------|----------|------------|
| 1 | 图像旋转 | 仿射变换 + 双线性插值 | O(H_out * W_out) |
| 2 | GMM分割 | EM算法 (E步+M步) | O(N*K*iter) |
| 3 | NMS | 贪心/Soft-NMS | O(N^2) 最坏 |
| 4 | 图像拼接 | Harris + DLT + RANSAC | O(N*M*k_iter) |
| 5 | 金字塔融合 | Gaussian/Laplacian金字塔 | O(N) |
| 6 | HOG特征 | 梯度 + Cell直方图 + Block归一化 | O(N*n_bins) |
| 7 | K-Means分割 | K-Means++初始化 + 迭代优化 | O(N*K*iter*d) |
| 8 | 光流估计 | Lucas-Kanade (局部窗口) | O(H*W) |
| 9 | 超分辨率 | 双三次插值核函数 | O(H_out*W_out*16) |
| 10 | OCR预处理 | 灰度化+二值化+去噪+倾斜校正+分割 | 多步骤流水线 |